In [1]:
# must match the hist_set_method the solves use, so the threshold below is
# evaluated at the n_ell the stability constraints actually enforce
HIST_SET_METHOD = "instance_farmers"
from stable_platform_matchings.domain.hist_sets import status_quo_quantities_for_method

from __future__ import annotations

import pickle
import platform
import sys
from pathlib import Path
from typing import Any

import numpy as np

from stable_platform_matchings import Optimizer
from stable_platform_matchings.optimization.options import OptimizerParams, SolverOptions
from stable_platform_matchings.domain.instance import Instance
from stable_platform_matchings.graphs.road_graphs import RoadGraph
import stable_platform_matchings.experiments.utils as utils


BASE_SEED = 20260918
VRP_TIME_LIMIT_SECONDS = 900

N_SAMPLES = 100
EPSILON_ELL = 0.5
TOP_N = [1, 2, 3]

B = 14

def set_epsilons(
    instance: Instance,
    treatment_ids: set,
    epsilon_h: float,
    epsilon_ell: float,
) -> dict[str, float]:
    """Sample an epsilon for every intermediary."""
    epsilons = {}
    for intermediary in instance.intermediaries:
        if intermediary.id in treatment_ids:
            epsilons[intermediary.id] = epsilon_h
        else:
            epsilons[intermediary.id] = epsilon_ell
    return epsilons


def set_het_costs(
    instance: Instance,
    treatment_ids: set,
    margin: float = 1
) -> dict[str, float]:
    """Sample heterogeneous costs for every intermediary."""
    base_het_costs = {
        intermediary.id: float(2.0 * instance.dist_to_mill[intermediary.id])
        for intermediary in instance.intermediaries
    }

    # get min treatment and max control sigmas
    min_treatment_sigma = min(
        het_cost for intermediary_id, het_cost in base_het_costs.items()
        if intermediary_id in treatment_ids
    )
    max_control_sigma = max(
        het_cost for intermediary_id, het_cost in base_het_costs.items()
        if intermediary_id not in treatment_ids
    )

    # scale treatment sigmas
    scale_factor = max(1, (max_control_sigma + margin) / min_treatment_sigma)

    het_costs = {}
    for intermediary_id in base_het_costs:
        if intermediary_id in treatment_ids:
            het_costs[intermediary_id] = base_het_costs[intermediary_id] * scale_factor
        else:
            het_costs[intermediary_id] = base_het_costs[intermediary_id]

    return het_costs

def clip_het_costs(
    instance: Instance,
    treatment_ids: set,
    margin: float = 100
) -> dict[str, float]:
    """Sample heterogeneous costs for every intermediary."""
    base_het_costs = {
        intermediary.id: float(2.0 * instance.dist_to_mill[intermediary.id])
        for intermediary in instance.intermediaries
    }

    # get min treatment and max control sigmas
    max_control_sigma = max(
        het_cost for intermediary_id, het_cost in base_het_costs.items()
        if intermediary_id not in treatment_ids
    )

    het_costs = {}
    for intermediary_id in base_het_costs:
        if intermediary_id in treatment_ids:
            het_costs[intermediary_id] = max(base_het_costs[intermediary_id], max_control_sigma + margin)
        else:
            het_costs[intermediary_id] = base_het_costs[intermediary_id]

    return het_costs

In [2]:

# get data paths
data_path = Path("../data")
instances_path = data_path / "anon_14_day_instances"
graph_path = data_path / "graph_0-14960_00_new.pickle"

# check if data is well-formed
if not instances_path.is_dir():
    raise FileNotFoundError(f"Instance directory does not exist: {instances_path}")
if not graph_path.is_file():
    raise FileNotFoundError(f"Graph file does not exist: {graph_path}")

# load instance paths
instance_paths = sorted(
    path for path in instances_path.iterdir()
    if path.is_file()
    and path.suffix.lower() in {".yaml", ".yml"}
    and not path.name.startswith("aggregate")
)
if not instance_paths:
    raise FileNotFoundError(f"No YAML instance files found in {instances_path}")

with graph_path.open("rb") as file:
    graph = pickle.load(file)

holds = []
checks=[]

for b in range(14):
    job_id = b
    rng = np.random.default_rng(seed=int(job_id))
    epsilon_hs = rng.uniform(0, 9, size=N_SAMPLES)
    top_n_idx = np.random.randint(0, 3)

    # load instance
    print("Loading instance...")
    instance_path = instance_paths[b]
    instance = Instance.from_yaml(instance_path)

    print(f"Loaded instance {instance_path}, setting graph...")
    instance.set_graph(RoadGraph(graph))

    top_n = TOP_N[top_n_idx]
    status_quo_sorted = sorted(
        status_quo_quantities_for_method(instance, HIST_SET_METHOD).items(),
        key=lambda x: x[1], 
        reverse=True
    )
    treatment_ids = {item[0] for item in status_quo_sorted[:top_n]}

    check = False

    for epsilon_h in epsilon_hs:

        sigmas = set_het_costs(instance, treatment_ids)

        high_types = {
            intermediary_id: sigmas[intermediary_id]
            for intermediary_id in treatment_ids
        }

        low_types = {
            intermediary.id for intermediary in instance.intermediaries
            if intermediary.id not in treatment_ids
        }

        r_sigma_h = np.random.choice(list(high_types.values()))

        r_ell = np.random.choice(list(low_types))
        r_sigma_ell, r_n_ell = sigmas[r_ell], status_quo_quantities_for_method(instance, HIST_SET_METHOD)[r_ell]

        rhs = (r_sigma_h / r_sigma_ell * (r_n_ell + EPSILON_ELL))

        if epsilon_h > rhs:
            check = True

        holds.append(rhs)
    checks.append(check)

np.mean(checks)

Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-27.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-28.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-29.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-30.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-31.yaml, setting graph...
Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-09-01.yaml, setting graph...


KeyboardInterrupt: 

In [ ]:

# get data paths
data_path = Path("../data")
instances_path = data_path / "anon_14_day_instances"
graph_path = data_path / "graph_0-14960_00_new.pickle"

# check if data is well-formed
if not instances_path.is_dir():
    raise FileNotFoundError(f"Instance directory does not exist: {instances_path}")
if not graph_path.is_file():
    raise FileNotFoundError(f"Graph file does not exist: {graph_path}")

# load instance paths
instance_paths = sorted(
    path for path in instances_path.iterdir()
    if path.is_file()
    and path.suffix.lower() in {".yaml", ".yml"}
    and not path.name.startswith("aggregate")
)
if not instance_paths:
    raise FileNotFoundError(f"No YAML instance files found in {instances_path}")

with graph_path.open("rb") as file:
    graph = pickle.load(file)

holds = []
checks=[]

for b in range(14):
    job_id = b
    rng = np.random.default_rng(seed=int(job_id))
    epsilon_hs = rng.uniform(0, 9, size=N_SAMPLES)

    top_n_idx = np.random.randint(0, 3)

    # load instance
    print("Loading instance...")
    instance_path = instance_paths[b]
    instance = Instance.from_yaml(instance_path)

    print(f"Loaded instance {instance_path}, setting graph...")
    instance.set_graph(RoadGraph(graph))

    top_n = TOP_N[top_n_idx]
    status_quo_sorted = sorted(
        status_quo_quantities_for_method(instance, HIST_SET_METHOD).items(),
        key=lambda x: x[1], 
        reverse=True
    )
    treatment_ids = {item[0] for item in status_quo_sorted[:top_n]}

    check = False

    for epsilon_h in epsilon_hs:

        sigmas = clip_het_costs(instance, treatment_ids)

        high_types = {
            intermediary_id: sigmas[intermediary_id]
            for intermediary_id in treatment_ids
        }

        low_types = {
            intermediary.id for intermediary in instance.intermediaries
            if intermediary.id not in treatment_ids
        }

        r_sigma_h = np.random.choice(list(high_types.values()))

        r_ell = np.random.choice(list(low_types))
        r_sigma_ell, r_n_ell = sigmas[r_ell], status_quo_quantities_for_method(instance, HIST_SET_METHOD)[r_ell]

        rhs = (r_sigma_h / r_sigma_ell * (r_n_ell + EPSILON_ELL))

        if epsilon_h > rhs:
            check = True

        holds.append(rhs)
    checks.append(check)

np.mean(checks)

Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-27.yaml, setting graph...


Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-28.yaml, setting graph...


Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-29.yaml, setting graph...


Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-30.yaml, setting graph...


Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-08-31.yaml, setting graph...


...

Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-09-06.yaml, setting graph...


Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-09-07.yaml, setting graph...


Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-09-08.yaml, setting graph...


Loading instance...
Loaded instance ../data/anon_14_day_instances/2020-09-09.yaml, setting graph...


np.float64(1.0)

In [3]:
import math
import pandas as pd


def prop3_thresholds(*, n_ell, sigma_ell, T_ell, n_h, sigma_h, T_h, epsilon_ell, K, F_total):
    """
    Evaluate Proposition 3's ambiguity thresholds (eps_h_minus, eps_h_plus, eps_h_bar) for a
    two-type stylized model with the given aggregate parameters.

    eps_h_minus/eps_h_plus bound the inefficient-matching interval in the n_h+eps_h <= K
    regime (cases ii-iii, root of EC3); eps_h_bar is the single threshold in the
    n_h+eps_h > K regime (cases iv-v, EC4). A value of None means that threshold's
    interval is empty for these parameters (no inefficient matching predicted in that
    regime); sigma_h <= sigma_ell means case (i) applies and no interval exists at all.
    """
    out = {
        "eps_h_minus": None,
        "eps_h_plus": None,
        "eps_h_bar": None,
        "regime_cutoff": K - n_h,
        "sigma_h_le_sigma_ell": sigma_h <= sigma_ell,
    }
    if sigma_h <= sigma_ell or T_h <= 0 or T_ell <= 0:
        return out

    e_ell = sigma_ell / (n_ell + epsilon_ell)

    # regime n_h + eps_h <= K: cases (ii)-(iii), roots of (EC3)
    M = T_ell * n_ell / T_h
    ratio = sigma_h / sigma_ell
    if ratio > 1:
        sqrt_term = math.sqrt(ratio) - math.sqrt(ratio - 1)
        eps_ell_u = M * sqrt_term**2 - n_ell
        out["eps_ell_u"] = eps_ell_u
        if epsilon_ell < eps_ell_u:
            b = -(M + n_ell + epsilon_ell)
            c = M * ratio * (n_ell + epsilon_ell)
            disc = b**2 - 4 * c
            if disc >= 0:
                root1 = (-b - math.sqrt(disc)) / 2
                root2 = (-b + math.sqrt(disc)) / 2
                out["eps_h_minus"], out["eps_h_plus"] = min(root1, root2), max(root1, root2)

    # regime n_h + eps_h > K: cases (iv)-(v), eps_h_bar from (EC4)
    denom = F_total * sigma_h - T_h * K * sigma_ell
    if denom > 0:
        eps_ell_s = K * (F_total - T_h * K) * sigma_ell / denom - n_ell
        out["eps_ell_s"] = eps_ell_s
        if epsilon_ell < eps_ell_s:
            cand1 = (K - n_h) + n_h * (sigma_h - sigma_ell) / (K * e_ell - sigma_ell)
            cand2 = T_ell * n_ell * sigma_h / ((F_total - T_h * K) * e_ell + T_h * sigma_ell)
            out["eps_h_bar"] = min(cand1, cand2)
        else:
            out["eps_h_bar"] = float("inf")

    return out


In [4]:
# Evaluate Prop 3 thresholds against the real 14 daily instances. sigma_t uses
# clip_het_costs (same convention as the check above: the heterogeneous cost alone,
# not adding instance.truck_fixed_cost, since that is how exp_7/8 already build
# het_costs and how the earlier check cell in this notebook treats sigma). n_t uses
# status_quo_quantities_for_method under HIST_SET_METHOD, matching the n_ell the
# stability constraints actually enforce. epsilon_ell is fixed at EPSILON_ELL, as in
# exp_7/exp_8.
#
# Two versions, per the "both side by side" choice:
#   - "aggregated": collapse the whole treated group into one representative high type
#     (T_h = top_n, sigma_h/n_h averaged over the treated group) -- most faithful to
#     Prop 3's literal two-homogeneous-type structure, but blurs real heterogeneity.
#   - "per_intermediary": treat each treated intermediary individually as the sole high
#     type (T_h = 1), pooling only the actual control (non-treated) intermediaries as
#     the low type. Other treated intermediaries are excluded from that intermediary's
#     calculation entirely (they are neither low- nor high-type in this local view).
K = instance.truck_capacity_tons if "instance" in dir() else 9.0

rows = []
for b, instance_path in enumerate(instance_paths):
    instance = Instance.from_yaml(instance_path)
    instance.set_graph(RoadGraph(graph))

    n_t_all = status_quo_quantities_for_method(instance, HIST_SET_METHOD)
    F_total = sum(n_t_all.values())
    K = instance.truck_capacity_tons

    for top_n in TOP_N:
        status_quo_sorted = sorted(
            n_t_all.items(), key=lambda item: (-item[1], item[0])
        )
        treatment_ids = {item[0] for item in status_quo_sorted[:top_n]}
        control_ids = {
            intermediary.id for intermediary in instance.intermediaries
            if intermediary.id not in treatment_ids
        }

        sigma_t_all = clip_het_costs(instance, treatment_ids)

        T_ell = len(control_ids)
        n_ell = sum(n_t_all[cid] for cid in control_ids) / T_ell
        sigma_ell = sum(sigma_t_all[cid] for cid in control_ids) / T_ell

        # aggregated version
        T_h = top_n
        n_h = sum(n_t_all[tid] for tid in treatment_ids) / T_h
        sigma_h = sum(sigma_t_all[tid] for tid in treatment_ids) / T_h
        agg = prop3_thresholds(
            n_ell=n_ell, sigma_ell=sigma_ell, T_ell=T_ell,
            n_h=n_h, sigma_h=sigma_h, T_h=T_h,
            epsilon_ell=EPSILON_ELL, K=K, F_total=F_total,
        )
        rows.append({
            "instance_idx": b, "top_n": top_n, "version": "aggregated",
            "intermediary_id": None,
            "n_h": n_h, "sigma_h": sigma_h, "n_ell": n_ell, "sigma_ell": sigma_ell,
            **agg,
        })

        # per-intermediary version
        for tid in treatment_ids:
            single = prop3_thresholds(
                n_ell=n_ell, sigma_ell=sigma_ell, T_ell=T_ell,
                n_h=n_t_all[tid], sigma_h=sigma_t_all[tid], T_h=1,
                epsilon_ell=EPSILON_ELL, K=K, F_total=F_total,
            )
            rows.append({
                "instance_idx": b, "top_n": top_n, "version": "per_intermediary",
                "intermediary_id": tid,
                "n_h": n_t_all[tid], "sigma_h": sigma_t_all[tid],
                "n_ell": n_ell, "sigma_ell": sigma_ell,
                **single,
            })

prop3_df = pd.DataFrame(rows)
prop3_df


,instance_idx,top_n,version,intermediary_id,n_h,sigma_h,n_ell,sigma_ell,eps_h_minus,eps_h_plus,eps_h_bar,regime_cutoff,sigma_h_le_sigma_ell,eps_ell_u,eps_ell_s
0,0,1,aggregated,NaN,8.800000,597502.690537,2.830769,374155.177905,5.682087,34.448683,3.286260,0.200000,False,6.043991,2.330591
1,0,1,per_intermediary,elegant_mendel,8.800000,597502.690537,2.830769,374155.177905,5.682087,34.448683,3.286260,0.200000,False,6.043991,2.330591
2,0,2,aggregated,NaN,8.500000,597502.690537,2.383333,369069.788811,5.933651,11.249682,2.979985,0.500000,False,0.989375,2.066384
3,0,2,per_intermediary,elegant_mendel,8.800000,597502.690537,2.383333,369069.788811,5.050704,26.432630,2.767514,0.200000,False,4.362084,2.698136
4,0,2,per_intermediary,quizzical_elgamal,8.200000,597502.690537,2.383333,369069.788811,5.050704,26.432630,3.192457,0.800000,False,4.362084,2.698136
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,13,2,per_intermediary,elegant_gagarin,8.800000,498951.668030,2.475000,255657.301362,6.617847,26.057153,4.188033,0.200000,False,2.801327,1.661312
122,13,3,aggregated,NaN,8.433333,498951.668030,1.990909,268387.963283,NaN,NaN,3.339123,0.566667,False,-0.599277,1.001780
123,13,3,per_intermediary,loving_engelbart,8.700000,498951.668030,1.990909,268387.963283,5.316848,19.074061,2.492298,0.300000,False,2.183986,2.374914
124,13,3,per_intermediary,competent_mayer,7.800000,498951.668030,1.990909,268387.963283,5.316848,19.074061,2.492298,1.200000,False,2.183986,2.374914
